# Phase 5 — Gesture Classification: Systematic Fix Notebook

**Problem:** Original 5-class Transformer got Macro F1 < 0.40
**Strategy:** Try cumulative solutions, stop when F1 >= 0.55

| Solution | What it does |
|----------|-------------|
| **Sol 1** | Collapse 5 classes → 3 (Active, Adaptor, Rest) |
| **Sol 2** | Drop z-coord, body-normalize, temporal smooth |
| **Sol 3** | Smaller models (TinyTransformer + BiLSTM) |
| **Sol 4** | Data-driven label thresholds + confidence weighting |

**Pose data:** Loaded from Drive cache (no video download needed)
**Setup:** Runtime → Change runtime type → **T4 GPU**

## Cell 1: Install Dependencies & GPU Check

In [ ]:
!pip install -q torch torchvision
!pip install -q scikit-learn tqdm scipy

import torch
import numpy as np
import os, time, json
from collections import Counter

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU! Runtime -> Change runtime type -> T4 GPU")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Cell 2: Mount Google Drive & Load Cached Poses

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/voice_pipeline_models/gesture_transformer"
POSE_CACHE_DIR = os.path.join(SAVE_DIR, "pose_cache")
os.makedirs(SAVE_DIR, exist_ok=True)

# Load all cached .npz pose files
all_video_poses = {}
npz_files = sorted([f for f in os.listdir(POSE_CACHE_DIR) if f.endswith('.npz')])
print(f"Found {len(npz_files)} cached pose files\n")

for fname in npz_files:
    fpath = os.path.join(POSE_CACHE_DIR, fname)
    data = np.load(fpath, allow_pickle=True)
    poses = {}
    none_indices = set()
    if 'none_indices' in data:
        none_indices = set(data['none_indices'].tolist())
    for key in data.files:
        if key == 'none_indices':
            continue
        idx = int(key.replace('frame_', ''))
        poses[idx] = data[key]
    for idx in none_indices:
        poses[int(idx)] = None
    vname = fname.replace('.npz', '')
    all_video_poses[vname] = poses
    detected = sum(1 for v in poses.values() if v is not None)
    print(f"  {vname}: {len(poses)} frames, {detected} with pose")

total_frames = sum(len(p) for p in all_video_poses.values())
total_detected = sum(
    sum(1 for v in p.values() if v is not None) for p in all_video_poses.values()
)
print(f"\nTotal: {total_frames} frames, {total_detected} with pose "
      f"({100*total_detected/max(total_frames,1):.1f}%)")
print(f"Videos: {len(all_video_poses)}")

## Cell 3: Shared Utilities

Constants, heuristic labeler, feature processing, dataset, models, training loop.

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from scipy.ndimage import uniform_filter1d

# ═══════════════════ Constants ═══════════════════
WINDOW_FRAMES = 15
HOP_FRAMES = 8
MIN_VALID_RATIO = 0.8
NUM_KEYPOINTS = 33
TARGET_FPS = 5
BATCH_SIZE = 64
NUM_EPOCHS = 30
PATIENCE = 10

# Body landmark indices
L_WRIST, R_WRIST = 15, 16
L_SHOULDER, R_SHOULDER = 11, 12
NOSE = 0
L_HIP, R_HIP = 23, 24

# Results tracker
results_table = []

# ═══════════════════ 5-Class Heuristic ═══════════════════
def heuristic_label_5class(kps_window):
    """Original heuristic. Input: (15, 33, 4). Returns (class_idx, confidence)."""
    l_wrist = kps_window[:, L_WRIST, :2]
    r_wrist = kps_window[:, R_WRIST, :2]
    shoulder_mid = (kps_window[:, L_SHOULDER, :2] + kps_window[:, R_SHOULDER, :2]) / 2
    hip_mid = (kps_window[:, L_HIP, :2] + kps_window[:, R_HIP, :2]) / 2
    nose = kps_window[:, NOSE, :2]

    l_delta = np.sqrt(np.sum(np.diff(l_wrist, axis=0) ** 2, axis=1))
    r_delta = np.sqrt(np.sum(np.diff(r_wrist, axis=0) ** 2, axis=1))
    total_movement = np.sum(l_delta) + np.sum(r_delta)

    l_face_dist = np.mean(np.sqrt(np.sum((l_wrist - nose) ** 2, axis=1)))
    r_face_dist = np.mean(np.sqrt(np.sum((r_wrist - nose) ** 2, axis=1)))
    min_face_dist = min(l_face_dist, r_face_dist)

    l_below = np.mean(l_wrist[:, 1] > hip_mid[:, 1])
    r_below = np.mean(r_wrist[:, 1] > hip_mid[:, 1])
    l_above = np.mean(l_wrist[:, 1] < shoulder_mid[:, 1])
    r_above = np.mean(r_wrist[:, 1] < shoulder_mid[:, 1])

    combined = l_delta + r_delta
    beat_score = 0.0
    if len(combined) > 3 and np.std(combined) > 1e-6:
        normed = (combined - np.mean(combined)) / (np.std(combined) + 1e-8)
        autocorr = np.correlate(normed, normed, mode='full')
        autocorr = autocorr[len(autocorr) // 2:]
        if len(autocorr) > 3:
            autocorr = autocorr / (autocorr[0] + 1e-8)
            beat_score = np.max(autocorr[2:min(7, len(autocorr))])

    symmetry = 0.0
    if np.sum(l_delta) > 0.01 and np.sum(r_delta) > 0.01:
        min_len = min(len(l_delta), len(r_delta))
        if np.std(l_delta[:min_len]) > 0 and np.std(r_delta[:min_len]) > 0:
            symmetry = np.corrcoef(l_delta[:min_len], r_delta[:min_len])[0, 1]

    if min_face_dist < 0.08 and total_movement > 0.05:
        return 3, 0.75
    if total_movement < 0.12:
        return 4, 0.85
    if total_movement < 0.20 and (l_below > 0.7 and r_below > 0.7):
        return 4, 0.75
    if beat_score > 0.35 and symmetry > 0.3 and total_movement > 0.15:
        return 2, 0.65
    if symmetry > 0.5 and (l_above > 0.3 or r_above > 0.3):
        return 1, 0.55
    if total_movement > 0.15:
        return 0, 0.60
    return 4, 0.50


# ═══════════════════ Window Creation ═══════════════════
def create_raw_windows(poses_dict):
    """Create sliding windows. Returns list of (15, 33, 4) arrays."""
    indices = sorted(poses_dict.keys())
    if not indices:
        return []
    max_idx = indices[-1]
    windows = []
    for start in range(indices[0], max_idx - WINDOW_FRAMES + 2, HOP_FRAMES):
        window_kps = []
        valid = 0
        for i in range(start, start + WINDOW_FRAMES):
            kp = poses_dict.get(i)
            if kp is not None:
                window_kps.append(kp)
                valid += 1
            else:
                window_kps.append(np.zeros((NUM_KEYPOINTS, 4), dtype=np.float32))
        if valid >= MIN_VALID_RATIO * WINDOW_FRAMES:
            windows.append(np.array(window_kps, dtype=np.float32))
    return windows


# ═══════════════════ Feature Processing (Sol 2) ═══════════════════
def drop_z(window_raw):
    """(15, 33, 4) -> (15, 33, 3): keep x, y, visibility."""
    return window_raw[:, :, [0, 1, 3]]


def normalize_to_body(kp_frame):
    """Normalize single frame (33, 3) to body-relative coords."""
    kp = kp_frame.copy()
    ls, rs = kp[L_SHOULDER, :2], kp[R_SHOULDER, :2]
    lh, rh = kp[L_HIP, :2], kp[R_HIP, :2]
    center = (ls + rs + lh + rh) / 4.0
    body_h = np.linalg.norm((ls + rs) / 2 - (lh + rh) / 2)
    if body_h < 0.01:
        return kp
    kp[:, 0] = (kp[:, 0] - center[0]) / body_h
    kp[:, 1] = (kp[:, 1] - center[1]) / body_h
    return kp


def normalize_window(w3d):
    """Normalize all frames in (15, 33, 3)."""
    return np.array([normalize_to_body(w3d[t]) for t in range(w3d.shape[0])])


def smooth_window(wf, kernel_size=3):
    """Smooth (15, D) along time axis."""
    return uniform_filter1d(wf, size=kernel_size, axis=0).astype(np.float32)


def process_windows_full(raw_windows):
    """Apply full feature pipeline: drop z + normalize + smooth."""
    processed = []
    for raw_w in raw_windows:
        w3 = drop_z(raw_w)
        w3 = normalize_window(w3)
        wf = w3.reshape(WINDOW_FRAMES, -1)
        wf = smooth_window(wf)
        processed.append(wf)
    return np.array(processed, dtype=np.float32)


# ═══════════════════ Dataset ═══════════════════
class GestureDataset(Dataset):
    def __init__(self, windows, labels, augment=False, confidences=None):
        self.windows = windows
        self.labels = labels
        self.augment = augment
        self.confidences = confidences

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        x = self.windows[idx].copy()
        y = self.labels[idx]
        if self.augment:
            x = self._augment(x)
        out = (torch.tensor(x, dtype=torch.float32), y)
        if self.confidences is not None:
            return (*out, torch.tensor(self.confidences[idx], dtype=torch.float32))
        return out

    def _augment(self, x):
        T, D = x.shape
        kp_dim = D // NUM_KEYPOINTS
        if np.random.random() < 0.5:
            x = x + np.random.normal(0, 0.01, x.shape).astype(np.float32)
        if np.random.random() < 0.3:
            x = np.roll(x, np.random.randint(-2, 3), axis=0)
        if np.random.random() < 0.5 and kp_dim > 0:
            xr = x.reshape(T, NUM_KEYPOINTS, kp_dim)
            xr[:, :, 0] = -xr[:, :, 0]
            lr = [(1,4),(2,5),(3,6),(7,8),(9,10),(11,12),(13,14),
                  (15,16),(17,18),(19,20),(21,22),(23,24),(25,26),
                  (27,28),(29,30),(31,32)]
            for l, r in lr:
                xr[:, [l, r]] = xr[:, [r, l]]
            x = xr.reshape(T, -1)
        if np.random.random() < 0.3:
            x = x * np.random.uniform(0.9, 1.1)
        return x.astype(np.float32)


# ═══════════════════ Models ═══════════════════
class GestureTransformer(nn.Module):
    """Original large Transformer (~3.8M params)."""
    def __init__(self, input_dim=132, num_classes=5):
        super().__init__()
        d = 256
        self.proj = nn.Linear(input_dim, d)
        self.pe = nn.Parameter(torch.randn(1, 64, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 4, 512, 0.1, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, 4)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, num_classes)

    def forward(self, x):
        x = self.proj(x)
        x = x + self.pe[:, :x.size(1)]
        x = self.norm(self.enc(x)).mean(dim=1)
        return self.head(x)


class TinyTransformer(nn.Module):
    """Small Transformer (~350K params)."""
    def __init__(self, input_dim=99, num_classes=3):
        super().__init__()
        d = 128
        self.proj = nn.Linear(input_dim, d)
        self.pe = nn.Parameter(torch.randn(1, 64, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 4, 256, 0.2, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, 2)
        self.head = nn.Sequential(
            nn.Linear(d, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, num_classes))

    def forward(self, x):
        x = self.proj(x)
        x = x + self.pe[:, :x.size(1)]
        x = self.enc(x).mean(dim=1)
        return self.head(x)


class GestureBiLSTM(nn.Module):
    """BiLSTM (~200K params)."""
    def __init__(self, input_dim=99, hidden=64, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, 2, batch_first=True,
                            bidirectional=True, dropout=0.2)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes))

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out.mean(dim=1))


# ═══════════════════ Training ═══════════════════
def train_and_evaluate(model, train_w, train_l, val_w, val_l, class_names,
                       lr=5e-4, use_conf=False, train_conf=None):
    """Train model, return (best_f1, per_class_dict, trained_model)."""
    model = model.to(device)
    nc = len(class_names)

    train_ds = GestureDataset(train_w, train_l, augment=True,
                              confidences=train_conf if use_conf else None)
    val_ds = GestureDataset(val_w, val_l, augment=False)

    drop_last = len(train_ds) >= BATCH_SIZE
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              drop_last=drop_last, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    if len(train_loader) == 0:
        print("  ERROR: No training batches!")
        return 0.0, {c: 0.0 for c in class_names}, model

    counts = Counter(train_l.tolist())
    n = len(train_l)
    weights = torch.tensor(
        [n / (nc * counts.get(i, 1)) for i in range(nc)], dtype=torch.float32
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=weights,
                                    reduction='none' if use_conf else 'mean')
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=len(train_loader) * NUM_EPOCHS)

    best_f1, best_state, pat = 0.0, None, 0

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        eloss = 0.0
        for batch in train_loader:
            if use_conf:
                x, y, c = batch
                x, y, c = x.to(device), y.to(device), c.to(device)
            else:
                x, y = batch
                x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            if use_conf:
                loss = (criterion(logits, y) * c).mean()
            else:
                loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            eloss += loss.item()
        eloss /= len(train_loader)

        model.eval()
        pa, ta = [], []
        with torch.no_grad():
            for batch in val_loader:
                x = batch[0].to(device)
                pa.extend(model(x).argmax(-1).cpu().tolist())
                ta.extend(batch[1].tolist())
        vf1 = f1_score(ta, pa, average='macro', zero_division=0)

        tag = ""
        if vf1 > best_f1:
            best_f1 = vf1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            pat = 0
            tag = " << BEST"
        else:
            pat += 1
            tag = f" ({pat}/{PATIENCE})"

        if epoch % 5 == 0 or "BEST" in tag:
            print(f"  Ep {epoch:2d}/{NUM_EPOCHS} | Loss {eloss:.4f} | F1 {vf1:.4f}{tag}")

        if pat >= PATIENCE:
            print(f"  Early stop at epoch {epoch}")
            break

    if best_state:
        model.load_state_dict(best_state)
    model.eval()
    pa, ta = [], []
    with torch.no_grad():
        for batch in val_loader:
            x = batch[0].to(device)
            pa.extend(model(x).argmax(-1).cpu().tolist())
            ta.extend(batch[1].tolist())

    report = classification_report(ta, pa, target_names=class_names,
                                   output_dict=True, zero_division=0)
    per_class = {c: round(report[c]['f1-score'], 4) for c in class_names}
    print(f"\n  Best Macro F1: {best_f1:.4f}")
    print(classification_report(ta, pa, target_names=class_names, zero_division=0))

    cm = confusion_matrix(ta, pa)
    header = "              " + "".join(f"{c[:12]:>14s}" for c in class_names)
    print(header)
    for i, c in enumerate(class_names):
        print(f"  {c:12s}" + "".join(f"{cm[i][j]:14d}" for j in range(nc)))

    return best_f1, per_class, model


# ═══════════════════ Reporting ═══════════════════
def report_solution(name, nc, model_name, params, f1, per_class,
                    n_train, n_val, dist):
    results_table.append(dict(name=name, nc=nc, model=model_name,
                              params=f"{params:,}", f1=f1))
    print(f"\n{'='*70}")
    print(f"SOLUTION RESULTS: {name}")
    print(f"{'='*70}")
    print(f"  Classes: {nc}")
    print(f"  Model: {model_name} ({params:,} params)")
    print(f"  Train samples: {n_train} | Val samples: {n_val}")
    print(f"  Class distribution: {dist}")
    print(f"  Val Macro F1: {f1:.4f}")
    print(f"  Per-class F1: {per_class}")
    if len(results_table) > 1:
        print(f"  Improvement over baseline: +{f1 - results_table[0]['f1']:.4f}")
    print(f"  Continue to next solution: {'NO (target reached!)' if f1 >= 0.55 else 'YES'}")
    print_table()
    return f1 >= 0.55


def print_table():
    print(f"\n{'='*90}")
    print(f"{'Solution':<30} {'Cls':>3} {'Model':<20} {'Params':>10} {'F1':>7} {'Delta':>7}")
    print(f"{'-'*90}")
    base = results_table[0]['f1'] if results_table else 0
    for r in results_table:
        d = r['f1'] - base
        ds = f"+{d:.4f}" if d >= 0 else f"{d:.4f}"
        print(f"  {r['name']:<28} {r['nc']:>3} {r['model']:<20} "
              f"{r['params']:>10} {r['f1']:>7.4f} {ds:>7}")
    print(f"{'='*90}\n")


print("All utilities loaded.")

## Cell 4: Create Raw Windows & Apply 5-Class Heuristic Labels

Creates 3-second windows (15 frames at 5fps) with 1.6s hop.
Shows the original 5-class distribution for reference.

In [ ]:
all_raw_windows = []
all_labels_5 = []
all_confs_5 = []

for vname, poses in all_video_poses.items():
    wins = create_raw_windows(poses)
    for w in wins:
        label, conf = heuristic_label_5class(w)
        all_raw_windows.append(w)
        all_labels_5.append(label)
        all_confs_5.append(conf)
    print(f"  {vname}: {len(wins)} windows")

all_raw_windows = np.array(all_raw_windows, dtype=np.float32)  # (N, 15, 33, 4)
all_labels_5 = np.array(all_labels_5, dtype=np.int64)
all_confs_5 = np.array(all_confs_5, dtype=np.float32)

CLASS_NAMES_5 = ["Illustrator", "Emblem", "Beat", "Adaptor", "Rest"]
print(f"\nTotal windows: {len(all_raw_windows)}, shape: {all_raw_windows.shape}")
print(f"\n5-class distribution:")
counts_5 = Counter(all_labels_5.tolist())
for i, c in enumerate(CLASS_NAMES_5):
    n = counts_5.get(i, 0)
    print(f"  {c:15s}: {n:>5} ({100*n/len(all_labels_5):.1f}%)")
print(f"\nMean heuristic confidence: {np.mean(all_confs_5):.3f}")

## Solution 1: Collapse to 3 Classes

Merge Illustrator + Emblem + Beat → **Active Gesture**.
Keep Adaptor and Rest separate. These 3 classes are visually distinct
in keypoint space and still useful for coaching feedback.

In [ ]:
print("=" * 70)
print("SOLUTION 1: Collapse 5 classes -> 3 classes")
print("=" * 70)

# Remap: 0,1,2 (Illustrator, Emblem, Beat) -> 0 (Active), 3 -> 1 (Adaptor), 4 -> 2 (Rest)
REMAP = {0: 0, 1: 0, 2: 0, 3: 1, 4: 2}
CLASS_NAMES_3 = ["Active Gesture", "Adaptor", "Rest"]

labels_3 = np.array([REMAP[l] for l in all_labels_5], dtype=np.int64)
confs_3 = all_confs_5.copy()

print("\n3-class distribution:")
counts_3 = Counter(labels_3.tolist())
for i, c in enumerate(CLASS_NAMES_3):
    n = counts_3.get(i, 0)
    print(f"  {c:15s}: {n:>5} ({100*n/len(labels_3):.1f}%)")

# Flatten: (N, 15, 33, 4) -> (N, 15, 132)
windows_flat = all_raw_windows.reshape(len(all_raw_windows), WINDOW_FRAMES, -1)

# Train/val split
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(sss.split(windows_flat, labels_3))

sol1_tw = windows_flat[train_idx]
sol1_tl = labels_3[train_idx]
sol1_vw = windows_flat[val_idx]
sol1_vl = labels_3[val_idx]
print(f"\nTrain: {len(sol1_tw)} | Val: {len(sol1_vw)}")

# Train large Transformer with 3 classes
model_sol1 = GestureTransformer(input_dim=132, num_classes=3)
p1 = sum(p.numel() for p in model_sol1.parameters())
print(f"Model: GestureTransformer ({p1:,} params)\n")

f1_sol1, pc_sol1, model_sol1 = train_and_evaluate(
    model_sol1, sol1_tw, sol1_tl, sol1_vw, sol1_vl, CLASS_NAMES_3)

dist_sol1 = {c: counts_3.get(i, 0) for i, c in enumerate(CLASS_NAMES_3)}
stop = report_solution("Sol 1: 3-Class", 3, "Transformer 3.8M", p1,
                       f1_sol1, pc_sol1, len(sol1_tw), len(sol1_vw), dist_sol1)

# Track best model so far
best_model_state = {k: v.cpu().clone() for k, v in model_sol1.state_dict().items()}
best_info = {"name": "GestureTransformer", "input_dim": 132, "num_classes": 3,
             "f1": f1_sol1, "classes": CLASS_NAMES_3, "solution": "Sol 1",
             "feature_pipeline": "raw"}

## Solution 2: Fix Feature Quality (cumulative)

Three fixes applied on top of Solution 1:
- **Drop z-coordinate**: MediaPipe's monocular z is noisy, adds confusion
- **Body-normalize**: Center on torso, scale by shoulder-hip distance (camera-invariant)
- **Temporal smooth**: Moving average (k=3) to reduce frame-level jitter

Input dim changes from 132 to 99.

In [ ]:
if results_table[-1]['f1'] >= 0.55:
    print("Target already reached! Skipping Solution 2.")
    f1_sol2 = results_table[-1]['f1']
else:
    print("=" * 70)
    print("SOLUTION 2: Fix features (drop z + body-normalize + smooth)")
    print("=" * 70)

    # Process all windows
    processed_windows = process_windows_full(all_raw_windows)
    print(f"Processed shape: {processed_windows.shape}  (was {windows_flat.shape})")

    sol2_tw = processed_windows[train_idx]
    sol2_vw = processed_windows[val_idx]
    sol2_tl = labels_3[train_idx]
    sol2_vl = labels_3[val_idx]

    model_sol2 = GestureTransformer(input_dim=99, num_classes=3)
    p2 = sum(p.numel() for p in model_sol2.parameters())
    print(f"Model: GestureTransformer input_dim=99 ({p2:,} params)\n")

    f1_sol2, pc_sol2, model_sol2 = train_and_evaluate(
        model_sol2, sol2_tw, sol2_tl, sol2_vw, sol2_vl, CLASS_NAMES_3)

    stop = report_solution("Sol 2: +Features", 3, "Transformer 3.7M", p2,
                           f1_sol2, pc_sol2, len(sol2_tw), len(sol2_vw), dist_sol1)

    if f1_sol2 > best_info["f1"]:
        best_model_state = {k: v.cpu().clone() for k, v in model_sol2.state_dict().items()}
        best_info = {"name": "GestureTransformer", "input_dim": 99, "num_classes": 3,
                     "f1": f1_sol2, "classes": CLASS_NAMES_3, "solution": "Sol 2",
                     "feature_pipeline": "drop_z+body_norm+smooth"}

## Solution 3: Shrink the Model (cumulative)

3.8M params for ~2,500 windows = massive overfitting.
Try two smaller architectures and pick the winner:
- **TinyTransformer**: 2 layers, d=128 (~350K params)
- **BiLSTM**: 2-layer bidirectional LSTM (~200K params)

In [ ]:
if results_table[-1]['f1'] >= 0.55:
    print("Target already reached! Skipping Solution 3.")
else:
    print("=" * 70)
    print("SOLUTION 3: Smaller models (TinyTransformer + BiLSTM)")
    print("=" * 70)

    # Use processed features from Sol 2
    sol3_tw = processed_windows[train_idx]
    sol3_vw = processed_windows[val_idx]
    sol3_tl = labels_3[train_idx]
    sol3_vl = labels_3[val_idx]

    # 3A: TinyTransformer
    print("\n--- TinyTransformer ---")
    m_tiny = TinyTransformer(input_dim=99, num_classes=3)
    p_tiny = sum(p.numel() for p in m_tiny.parameters())
    print(f"Params: {p_tiny:,}\n")
    f1_tiny, pc_tiny, m_tiny = train_and_evaluate(
        m_tiny, sol3_tw, sol3_tl, sol3_vw, sol3_vl, CLASS_NAMES_3)

    # 3B: BiLSTM
    print("\n--- BiLSTM ---")
    m_lstm = GestureBiLSTM(input_dim=99, num_classes=3)
    p_lstm = sum(p.numel() for p in m_lstm.parameters())
    print(f"Params: {p_lstm:,}\n")
    f1_lstm, pc_lstm, m_lstm = train_and_evaluate(
        m_lstm, sol3_tw, sol3_tl, sol3_vw, sol3_vl, CLASS_NAMES_3)

    # Pick winner
    if f1_tiny >= f1_lstm:
        print(f"\nWinner: TinyTransformer (F1={f1_tiny:.4f} vs BiLSTM {f1_lstm:.4f})")
        sol3_f1, sol3_pc = f1_tiny, pc_tiny
        sol3_model, sol3_name, sol3_p = m_tiny, "TinyTransformer", p_tiny
    else:
        print(f"\nWinner: BiLSTM (F1={f1_lstm:.4f} vs TinyTransformer {f1_tiny:.4f})")
        sol3_f1, sol3_pc = f1_lstm, pc_lstm
        sol3_model, sol3_name, sol3_p = m_lstm, "BiLSTM", p_lstm

    stop = report_solution(f"Sol 3: {sol3_name}", 3, sol3_name, sol3_p,
                           sol3_f1, sol3_pc, len(sol3_tw), len(sol3_vw), dist_sol1)

    if sol3_f1 > best_info["f1"]:
        best_model_state = {k: v.cpu().clone() for k, v in sol3_model.state_dict().items()}
        best_info = {"name": sol3_name, "input_dim": 99, "num_classes": 3,
                     "f1": sol3_f1, "classes": CLASS_NAMES_3, "solution": "Sol 3",
                     "feature_pipeline": "drop_z+body_norm+smooth"}

## Solution 4: Improve the Heuristic Labeler (cumulative)

Three improvements:
- **Data-driven thresholds**: Use percentiles instead of arbitrary constants
- **Skip ambiguous samples**: Better 1,500 clean labels than 2,500 noisy ones
- **Confidence-weighted loss**: Weight each sample by labeler certainty

In [ ]:
if results_table[-1]['f1'] >= 0.55:
    print("Target already reached! Skipping Solution 4.")
else:
    print("=" * 70)
    print("SOLUTION 4: Improved heuristic labeler (data-driven thresholds)")
    print("=" * 70)

    # Compute features on BODY-NORMALIZED coordinates
    all_movements = []
    all_face_dists = []
    all_below_fracs = []

    for raw_w in all_raw_windows:
        w3 = drop_z(raw_w)
        w3 = normalize_window(w3)

        lw = w3[:, L_WRIST, :2]
        rw = w3[:, R_WRIST, :2]
        ns = w3[:, NOSE, :2]
        hm = (w3[:, L_HIP, :2] + w3[:, R_HIP, :2]) / 2

        ld = np.sqrt(np.sum(np.diff(lw, axis=0)**2, axis=1))
        rd = np.sqrt(np.sum(np.diff(rw, axis=0)**2, axis=1))
        mv = np.sum(ld) + np.sum(rd)

        lfd = np.mean(np.sqrt(np.sum((lw - ns)**2, axis=1)))
        rfd = np.mean(np.sqrt(np.sum((rw - ns)**2, axis=1)))
        mfd = min(lfd, rfd)

        lb = np.mean(lw[:, 1] > hm[:, 1])
        rb = np.mean(rw[:, 1] > hm[:, 1])
        bw = max(lb, rb)

        all_movements.append(mv)
        all_face_dists.append(mfd)
        all_below_fracs.append(bw)

    all_movements = np.array(all_movements)
    all_face_dists = np.array(all_face_dists)
    all_below_fracs = np.array(all_below_fracs)

    # Data-driven thresholds
    move_p25 = np.percentile(all_movements, 25)
    move_p50 = np.percentile(all_movements, 50)
    face_p15 = np.percentile(all_face_dists, 15)

    print(f"Movement 25th pct: {move_p25:.4f}")
    print(f"Movement 50th pct: {move_p50:.4f}")
    print(f"Face dist 15th pct: {face_p15:.4f}")

    # Re-label with data-driven thresholds; skip ambiguous windows
    new_labels = []
    new_confs = []
    keep_mask = np.ones(len(all_raw_windows), dtype=bool)

    for i in range(len(all_raw_windows)):
        mv = all_movements[i]
        fd = all_face_dists[i]
        bw = all_below_fracs[i]

        if fd < face_p15 and mv < move_p50:
            new_labels.append(1)   # Adaptor
            new_confs.append(0.80)
        elif mv < move_p25 and bw > 0.6:
            new_labels.append(2)   # Rest
            new_confs.append(0.85)
        elif mv > move_p50:
            new_labels.append(0)   # Active Gesture
            new_confs.append(0.75)
        else:
            new_labels.append(-1)  # Ambiguous — skip
            new_confs.append(0.0)
            keep_mask[i] = False

    new_labels = np.array(new_labels, dtype=np.int64)
    new_confs = np.array(new_confs, dtype=np.float32)

    kept = keep_mask.sum()
    skipped = (~keep_mask).sum()
    print(f"\nKept: {kept} windows, Skipped (ambiguous): {skipped}")

    # Filter
    filtered_windows = processed_windows[keep_mask]
    filtered_labels = new_labels[keep_mask]
    filtered_confs = new_confs[keep_mask]

    print(f"\nNew 3-class distribution:")
    new_counts = Counter(filtered_labels.tolist())
    for i, c in enumerate(CLASS_NAMES_3):
        n = new_counts.get(i, 0)
        print(f"  {c:15s}: {n:>5} ({100*n/len(filtered_labels):.1f}%)")

    # Re-split on filtered data
    sss4 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    tr4, va4 = next(sss4.split(filtered_windows, filtered_labels))

    sol4_tw = filtered_windows[tr4]
    sol4_tl = filtered_labels[tr4]
    sol4_tc = filtered_confs[tr4]
    sol4_vw = filtered_windows[va4]
    sol4_vl = filtered_labels[va4]

    # Use best architecture from Sol 3
    arch = best_info["name"]
    if "LSTM" in arch:
        model_sol4 = GestureBiLSTM(input_dim=99, num_classes=3)
    elif "Tiny" in arch:
        model_sol4 = TinyTransformer(input_dim=99, num_classes=3)
    else:
        model_sol4 = TinyTransformer(input_dim=99, num_classes=3)
        arch = "TinyTransformer"
    p4 = sum(p.numel() for p in model_sol4.parameters())
    print(f"\nModel: {arch} ({p4:,} params)")
    print("Training with confidence-weighted loss...\n")

    f1_sol4, pc_sol4, model_sol4 = train_and_evaluate(
        model_sol4, sol4_tw, sol4_tl, sol4_vw, sol4_vl,
        CLASS_NAMES_3, use_conf=True, train_conf=sol4_tc)

    dist_sol4 = {c: new_counts.get(i, 0) for i, c in enumerate(CLASS_NAMES_3)}
    stop = report_solution(f"Sol 4: +Labels ({arch})", 3, arch, p4,
                           f1_sol4, pc_sol4, len(sol4_tw), len(sol4_vw), dist_sol4)

    if f1_sol4 > best_info["f1"]:
        best_model_state = {k: v.cpu().clone() for k, v in model_sol4.state_dict().items()}
        best_info = {"name": arch, "input_dim": 99, "num_classes": 3,
                     "f1": f1_sol4, "classes": CLASS_NAMES_3, "solution": "Sol 4",
                     "feature_pipeline": "drop_z+body_norm+smooth"}

## Save Best Model to Google Drive

Saves the best model from all solutions tried, along with metadata.

In [ ]:
print("=" * 70)
print("SAVING BEST MODEL")
print("=" * 70)
print(f"\nBest: {best_info['solution']} -- {best_info['name']} -- F1={best_info['f1']:.4f}")

# Save model checkpoint
torch.save({
    'model_state_dict': best_model_state,
    'model_name': best_info['name'],
    'input_dim': best_info['input_dim'],
    'num_classes': best_info['num_classes'],
    'classes': best_info['classes'],
    'val_f1': best_info['f1'],
    'solution': best_info['solution'],
    'feature_pipeline': best_info['feature_pipeline'],
}, f"{SAVE_DIR}/best_model.pt")

# Save metadata
meta = {
    'val_f1': float(best_info['f1']),
    'model_name': best_info['name'],
    'input_dim': best_info['input_dim'],
    'num_classes': best_info['num_classes'],
    'classes': best_info['classes'],
    'solution': best_info['solution'],
    'feature_pipeline': best_info['feature_pipeline'],
    'total_windows': int(len(all_raw_windows)),
    'num_videos': len(all_video_poses),
    'window_frames': WINDOW_FRAMES,
    'hop_frames': HOP_FRAMES,
    'results_table': results_table,
}
with open(f"{SAVE_DIR}/training_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"\nSaved to {SAVE_DIR}/")
!ls -lh {SAVE_DIR}/best_model.pt {SAVE_DIR}/training_meta.json

final_f1 = best_info['f1']
if final_f1 >= 0.55:
    print(f"\nSUCCESS: F1 = {final_f1:.4f} >= 0.55")
elif final_f1 >= 0.45:
    print(f"\nPARTIAL: F1 = {final_f1:.4f}. Consider adding more videos (Solution 5).")
else:
    print(f"\nBELOW TARGET: F1 = {final_f1:.4f}")
    print("Recommendation: Use heuristic fallback in pipeline for v1.")

print("\nDownload best_model.pt to:")
print("  ~/Desktop/Claude-assistant/models/gesture_transformer/best_model.pt")

print_table()

## (Optional) Visualize Gesture Timeline on First Video

In [ ]:
import matplotlib.pyplot as plt

# Use processed features if available, else raw
use_processed = 'processed_windows' in dir() and processed_windows is not None

# Reconstruct model for inference
if best_info['name'] == 'BiLSTM':
    viz_model = GestureBiLSTM(input_dim=best_info['input_dim'],
                              num_classes=best_info['num_classes'])
elif best_info['name'] == 'TinyTransformer':
    viz_model = TinyTransformer(input_dim=best_info['input_dim'],
                                num_classes=best_info['num_classes'])
else:
    viz_model = GestureTransformer(input_dim=best_info['input_dim'],
                                   num_classes=best_info['num_classes'])
viz_model.load_state_dict(best_model_state)
viz_model = viz_model.to(device).eval()

# Get windows for first video
first_video = list(all_video_poses.keys())[0]
first_wins_raw = create_raw_windows(all_video_poses[first_video])

if use_processed and best_info['input_dim'] == 99:
    first_wins = process_windows_full(np.array(first_wins_raw))
else:
    first_wins = np.array(first_wins_raw).reshape(len(first_wins_raw), WINDOW_FRAMES, -1)

tensor = torch.tensor(first_wins, dtype=torch.float32).to(device)
with torch.no_grad():
    logits = viz_model(tensor)
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    preds = logits.argmax(-1).cpu().numpy()

times = np.arange(len(preds)) * HOP_FRAMES / TARGET_FPS
class_names = best_info['classes']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
colors = ['#2ecc71', '#f39c12', '#95a5a6']

for i, cls in enumerate(class_names):
    mask = preds == i
    if mask.any():
        ax1.scatter(times[mask], [i]*mask.sum(), c=colors[i], s=30, label=cls, alpha=0.7)
ax1.set_yticks(range(len(class_names)))
ax1.set_yticklabels(class_names)
ax1.set_ylabel('Gesture Class')
ax1.set_title(f'Gesture Timeline: {first_video}')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

for i, cls in enumerate(class_names):
    ax2.plot(times, probs[:, i], label=cls, color=colors[i], alpha=0.8)
ax2.set_xlabel('Time (seconds)')
ax2.set_ylabel('Probability')
ax2.set_title('Class Probabilities Over Time')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/gesture_timeline.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved timeline to {SAVE_DIR}/gesture_timeline.png")